# ChemicalEntity ↔ ChemicalEntity Relation-Wise Merge

Merges Chemical–Chemical triples from Monarch, DRKG, PrimeKG (×2), PharmKG, Hetionet,
CrossBAR, iBKH, DtiNet, STITCH, and pheknowlator; resolves chemical names via PubChem and DrugBank;
assigns `head_id_is` / `tail_id_is` based on ID prefix; deduplicates by `(head, relation, tail)`;
and saves the result.

## 0. Configuration

In [4]:
import pandas as pd
import numpy as np
import re

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_pheknowlator/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/'
# pheknowlator_DIR = PROC_DIR + '4pheknowlator/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CHEMICALENTITY_CHEMICALENTITY/ALL_CHEMICALENTITY_CHEMICALENTITY.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

- **PubChem** — CID → IUPAC name and SMILES (for numeric PubChem IDs)
- **DrugBank** — DrugBank ID → drug name (for `DB`-prefixed IDs)

In [5]:
# PubChem compound table: CID → IUPAC name and SMILES
Pubchem = pd.read_pickle(DB_DIR + 'databases_for_mapping/pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem

print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict: 119,177,440 entries


In [6]:
# DrugBank: DrugBank ID → drug name
Drugbank = pd.read_csv(DB_DIR + 'databases_for_mapping/drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))

print(f"DrugBank dict: {len(Drugbank_dict):,} entries")

DrugBank dict: 16,575 entries


## 2. Load KG Sources

### 2a. Monarch

In [14]:
Monarch_Chemical_Chemical = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Human/ChemicalEntity_ChemicalEntity.csv')
Monarch_Chemical_Chemical.columns = Monarch_Chemical_Chemical.columns.str.lower()
Monarch_Chemical_Chemical = Monarch_Chemical_Chemical.rename(columns={'kgsource': 'kg_source'})
print(f"Monarch:  {len(Monarch_Chemical_Chemical):,} rows | columns: {list(Monarch_Chemical_Chemical.columns)}")
Monarch_Chemical_Chemical = Monarch_Chemical_Chemical.rename(columns={
    'head_name': 'head_detail_name',
    'tail_name': 'tail_detail_name',
})

Monarch_Chemical_Chemical['kg_type'] = 'Generalised'
Monarch_Chemical_Chemical['kg_source'] = 'Monarch_KG'
Monarch_Chemical_Chemical['species'] = 'Homo sapiens'
Monarch_Chemical_Chemical

Monarch:  11,407 rows | columns: ['head', 'tail', 'relation_type', 'relation_source', 'kg_source', 'head_name', 'head_type', 'head_id_is', 'head_taxon', 'head_taxon_label', 'tail_name', 'tail_type', 'tail_id_is', 'tail_taxon', 'tail_taxon_label', 'head_taxon_name', 'tail_taxon_name', 'head_type_clean', 'tail_type_clean', 'relation', 'species_species', 'head_id', 'tail_id']


,head,tail,relation_type,relation_source,kg_source,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,head_taxon_name,tail_taxon_name,head_type_clean,tail_type_clean,relation,species_species,head_id,tail_id,kg_type,species
0,135398636,135823789,related_to,infores:chebi,Monarch_KG,"guanosine 3',5'-bis(diphosphate)(5-)",ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77828,CHEBI:58215,Generalised,Homo sapiens
1,135398636,135398637,related_to,infores:chebi,Monarch_KG,"guanosine 3',5'-bis(diphosphate)(5-)",ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77828,CHEBI:17633,Generalised,Homo sapiens
2,25201902,10100906,related_to,infores:chebi,Monarch_KG,delphinidin 3-O-beta-D-glucoside-5-O-beta-D-gl...,ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77838,CHEBI:55455,Generalised,Homo sapiens
3,86289381,54740357,related_to,infores:chebi,Monarch_KG,stipitatate(1-),ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77842,CHEBI:57587,Generalised,Homo sapiens
4,86289381,20501,related_to,infores:chebi,Monarch_KG,stipitatate(1-),ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77842,CHEBI:15957,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11402,86289372,128861,related_to,infores:chebi,Monarch_KG,cyanidin 3-O-(2-O-beta-D-glucuronosyl)-beta-D-...,ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77824,CHEBI:27843,Generalised,Homo sapiens
11403,86289372,50909829,related_to,infores:chebi,Monarch_KG,cyanidin 3-O-(2-O-beta-D-glucuronosyl)-beta-D-...,ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77824,CHEBI:61317,Generalised,Homo sapiens
11404,86289374,46878438,related_to,infores:chebi,Monarch_KG,luteolin 7-O-[(beta-D-glucosyluronate)-(1->2)-...,ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77825,CHEBI:58678,Generalised,Homo sapiens
11405,86289375,45266663,related_to,infores:chebi,Monarch_KG,mugineate(1-),ChemicalEntity,Pubchem,NaN,NaN,...,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77826,CHEBI:58505,Generalised,Homo sapiens


### 2b. DRKG

In [17]:
DRKG_Chemical_Chemical = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Chemical_Chemical.csv')
DRKG_Chemical_Chemical.columns = DRKG_Chemical_Chemical.columns.str.lower()
print(f"DRKG:     {len(DRKG_Chemical_Chemical):,} rows | columns: {list(DRKG_Chemical_Chemical.columns)}")
DRKG_Chemical_Chemical = DRKG_Chemical_Chemical.rename(columns={
    'head_name': 'head_detail_name',
    'tail_name': 'tail_detail_name',
})

DRKG_Chemical_Chemical['relation']  = 'ChemicalEntity_ChemicalEntity'
DRKG_Chemical_Chemical['head_type']  = 'ChemicalEntity'
DRKG_Chemical_Chemical['tail_type']  = 'ChemicalEntity'


DRKG_Chemical_Chemical['kg_type'] = 'Generalised'
DRKG_Chemical_Chemical['kg_source'] = 'DRKG'
DRKG_Chemical_Chemical['species'] = 'Homo sapiens'
DRKG_Chemical_Chemical

DRKG:     1,385,757 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id', 'head_id_is', 'tail_id_is', 'head_orignal', 'tail_orignal']


/tmp/ipykernel_2687068/1379379645.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  DRKG_Chemical_Chemical = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Chemical_Chemical.csv')


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,DB00001,ChemicalEntity_ChemicalEntity,10182969,ChemicalEntity,DRUGBANK::ddi-interactor-in::Compound:Compound,ChemicalEntity,DRKG,DB00001,Drugbank,Pubchem,Compound::DB00001,Compound::DB06605,Generalised,Homo sapiens
1,DB00001,ChemicalEntity_ChemicalEntity,135565674,ChemicalEntity,DRUGBANK::ddi-interactor-in::Compound:Compound,ChemicalEntity,DRKG,DB00001,Drugbank,Pubchem,Compound::DB00001,Compound::DB06695,Generalised,Homo sapiens
2,DB00001,ChemicalEntity_ChemicalEntity,3062316,ChemicalEntity,DRUGBANK::ddi-interactor-in::Compound:Compound,ChemicalEntity,DRKG,DB00001,Drugbank,Pubchem,Compound::DB00001,Compound::DB01254,Generalised,Homo sapiens
3,DB00001,ChemicalEntity_ChemicalEntity,214348,ChemicalEntity,DRUGBANK::ddi-interactor-in::Compound:Compound,ChemicalEntity,DRKG,DB00001,Drugbank,Pubchem,Compound::DB00001,Compound::DB01609,Generalised,Homo sapiens
4,DB00001,ChemicalEntity_ChemicalEntity,31401,ChemicalEntity,DRUGBANK::ddi-interactor-in::Compound:Compound,ChemicalEntity,DRKG,DB00001,Drugbank,Pubchem,Compound::DB00001,Compound::DB01586,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1385752,5340,ChemicalEntity_ChemicalEntity,9047,ChemicalEntity,Hetionet::CrC::Compound:Compound,ChemicalEntity,DRKG,DB06147,Pubchem,Pubchem,Compound::DB06147,Compound::DB00664,Generalised,Homo sapiens
1385753,4057,ChemicalEntity_ChemicalEntity,3494,ChemicalEntity,Hetionet::CrC::Compound:Compound,ChemicalEntity,DRKG,DB04843,Pubchem,Pubchem,Compound::DB04843,Compound::DB00986,Generalised,Homo sapiens
1385754,9878,ChemicalEntity_ChemicalEntity,31307,ChemicalEntity,Hetionet::CrC::Compound:Compound,ChemicalEntity,DRKG,DB00324,Pubchem,Pubchem,Compound::DB00324,Compound::DB00620,Generalised,Homo sapiens
1385755,3241,ChemicalEntity_ChemicalEntity,34312,ChemicalEntity,Hetionet::CrC::Compound:Compound,ChemicalEntity,DRKG,DB00751,Pubchem,Pubchem,Compound::DB00751,Compound::DB00776,Generalised,Homo sapiens


### 2c. PrimeKG

In [22]:
PrimeKG_Chemical_Chemical_1 = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Chemical_Chemical_1.csv')
PrimeKG_Chemical_Chemical_1.columns = PrimeKG_Chemical_Chemical_1.columns.str.lower()
PrimeKG_Chemical_Chemical_1['kg_type'] = 'Generalised'
PrimeKG_Chemical_Chemical_1['kg_source'] = 'PrimeKG'
PrimeKG_Chemical_Chemical_1['species'] = 'Homo sapiens'
print(f"PrimeKG 1: {len(PrimeKG_Chemical_Chemical_1):,} rows")
PrimeKG_Chemical_Chemical_1

PrimeKG 1: 385 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,head_pubchem_id,head_smiles_name,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,kg_type,species
0,37035,ChemicalEntity_ChemicalEntity,37037,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"1,2,3-trichloro-4-(2,4,5-trichlorophenyl)benzene","2,2',3',4,4',5-hexachlorobiphenyl",C1=CC(=C(C(=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl)Cl,"1,2,3,4-tetrachloro-5-(2,3,4-trichlorophenyl)b...","2,2',3,3',4,4',5-heptachlorobiphenyl",C1=CC(=C(C(=C1C2=CC(=C(C(=C2Cl)Cl)Cl)Cl)Cl)Cl)Cl,C1=CC(=C(C(=C1C2=CC(=C(C(=C2Cl)Cl)Cl)Cl)Cl)Cl)Cl,Generalised,Homo sapiens
1,6919034,ChemicalEntity_ChemicalEntity,37037,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"[(1S,5R)-8-methyl-8-azoniabicyclo[3.2.1]octan-...","2,3,3',4,4',5-hexachlorobiphenyl",CC1=CC(=CC(=C1)C(=O)OC2C[C@H]3CC[C@@H](C2)[NH+...,"1,2,3,4-tetrachloro-5-(2,3,4-trichlorophenyl)b...","2,2',3,3',4,4',5-heptachlorobiphenyl",C1=CC(=C(C(=C1C2=CC(=C(C(=C2Cl)Cl)Cl)Cl)Cl)Cl)Cl,C1=CC(=C(C(=C1C2=CC(=C(C(=C2Cl)Cl)Cl)Cl)Cl)Cl)Cl,Generalised,Homo sapiens
2,6919034,ChemicalEntity_ChemicalEntity,37035,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"[(1S,5R)-8-methyl-8-azoniabicyclo[3.2.1]octan-...","2,3,3',4,4',5-hexachlorobiphenyl",CC1=CC(=CC(=C1)C(=O)OC2C[C@H]3CC[C@@H](C2)[NH+...,"1,2,3-trichloro-4-(2,4,5-trichlorophenyl)benzene","2,2',3',4,4',5-hexachlorobiphenyl",C1=CC(=C(C(=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl)Cl,C1=CC(=C(C(=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl)Cl,Generalised,Homo sapiens
3,36188,ChemicalEntity_ChemicalEntity,38017,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"1,2,3-trichloro-4-(3,4-dichlorophenyl)benzene","2,3,3',4,4'-pentachlorobiphenyl",C1=CC(=C(C=C1C2=C(C(=C(C=C2)Cl)Cl)Cl)Cl)Cl,"1,2,3-trichloro-4-(2,3,6-trichlorophenyl)benzene","2,2',3,3',4,6'-hexachlorobiphenyl",C1=CC(=C(C(=C1C2=C(C=CC(=C2Cl)Cl)Cl)Cl)Cl)Cl,C1=CC(=C(C(=C1C2=C(C=CC(=C2Cl)Cl)Cl)Cl)Cl)Cl,Generalised,Homo sapiens
4,35823,ChemicalEntity_ChemicalEntity,37037,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"1,2,4-trichloro-5-(3,4-dichlorophenyl)benzene","2,3',4,4',5-pentachlorobiphenyl",C1=CC(=C(C=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl,"1,2,3,4-tetrachloro-5-(2,3,4-trichlorophenyl)b...","2,2',3,3',4,4',5-heptachlorobiphenyl",C1=CC(=C(C(=C1C2=CC(=C(C(=C2Cl)Cl)Cl)Cl)Cl)Cl)Cl,C1=CC(=C(C(=C1C2=CC(=C(C(=C2Cl)Cl)Cl)Cl)Cl)Cl)Cl,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380,854019,ChemicalEntity_ChemicalEntity,5564,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,(5S)-1-methyl-5-pyridin-3-ylpyrrolidin-2-one,Cotinine,CN1[C@@H](CCC1=O)C2=CN=CC=C2,"5-chloro-2-(2,4-dichlorophenoxy)phenol",Triclosan,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,Generalised,Homo sapiens
381,137319715,ChemicalEntity_ChemicalEntity,5564,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,2-imino-1-methylimidazolidin-4-one,Creatinine,CN1CC(=O)NC1=N,"5-chloro-2-(2,4-dichlorophenoxy)phenol",Triclosan,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,Generalised,Homo sapiens
382,71684354,ChemicalEntity_ChemicalEntity,5564,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"(4S)-3,4-bis[(3-hydroxyphenyl)methyl]oxolan-2-one","2,3-bis(3'-hydroxybenzyl)butyrolactone",C1[C@H](C(C(=O)O1)CC2=CC(=CC=C2)O)CC3=CC(=CC=C3)O,"5-chloro-2-(2,4-dichlorophenoxy)phenol",Triclosan,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,Generalised,Homo sapiens
383,8449,ChemicalEntity_ChemicalEntity,5564,ChemicalEntity,parent-child,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"2,4-dichlorophenol","2,4-dichlorophenol",C1=CC(=C(C=C1Cl)Cl)O,"5-chloro-2-(2,4-dichlorophenoxy)phenol",Triclosan,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,Generalised,Homo sapiens


In [24]:

PrimeKG_Chemical_Chemical_2 = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Chemical_Chemical_2.csv')
PrimeKG_Chemical_Chemical_2.columns = PrimeKG_Chemical_Chemical_2.columns.str.lower()
PrimeKG_Chemical_Chemical_2['kg_type'] = 'Generalised'
PrimeKG_Chemical_Chemical_2['kg_source'] = 'PrimeKG'
PrimeKG_Chemical_Chemical_2['species'] = 'Homo sapiens'
print(f"PrimeKG 2: {len(PrimeKG_Chemical_Chemical_2):,} rows")
PrimeKG_Chemical_Chemical_2

PrimeKG 2: 1,336,314 rows


/tmp/ipykernel_2687068/1738506097.py:1: DtypeWarning: Columns (2,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  PrimeKG_Chemical_Chemical_2 = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Chemical_Chemical_2.csv')


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,head_pubchem_id,head_smiles_name,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,kg_type,species
0,10182969,ChemicalEntity_ChemicalEntity,135565674,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Pubchem,Pubchem,1-(4-methoxyphenyl)-7-oxo-6-[4-(2-oxopiperidin...,Apixaban,COC1=CC=C(C=C1)N2C3=C(CCN(C3=O)C4=CC=C(C=C4)N5...,ethyl 3-[[2-[[4-(N-hexoxycarbonylcarbamimidoyl...,Dabigatran etexilate,CCCCCCOC(=O)NC(=N)C1=CC=C(C=C1)NCC2=NC3=C(N2C)...,CCCCCCOC(=O)NC(=N)C1=CC=C(C=C1)NCC2=NC3=C(N2C)...,Generalised,Homo sapiens
1,10182969,ChemicalEntity_ChemicalEntity,3062316,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Pubchem,Pubchem,1-(4-methoxyphenyl)-7-oxo-6-[4-(2-oxopiperidin...,Apixaban,COC1=CC=C(C=C1)N2C3=C(CCN(C3=O)C4=CC=C(C=C4)N5...,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,Dasatinib,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...,Generalised,Homo sapiens
2,135565674,ChemicalEntity_ChemicalEntity,3062316,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Pubchem,Pubchem,ethyl 3-[[2-[[4-(N-hexoxycarbonylcarbamimidoyl...,Dabigatran etexilate,CCCCCCOC(=O)NC(=N)C1=CC=C(C=C1)NCC2=NC3=C(N2C)...,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,Dasatinib,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...,Generalised,Homo sapiens
3,3062316,ChemicalEntity_ChemicalEntity,214348,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Pubchem,Pubchem,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,Dasatinib,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...,"4-[3,5-bis(2-hydroxyphenyl)-1,2,4-triazol-1-yl...",Deferasirox,C1=CC=C(C(=C1)C2=NN(C(=N2)C3=CC=CC=C3O)C4=CC=C...,C1=CC=C(C(=C1)C2=NN(C(=N2)C3=CC=CC=C3O)C4=CC=C...,Generalised,Homo sapiens
4,10182969,ChemicalEntity_ChemicalEntity,214348,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Pubchem,Pubchem,1-(4-methoxyphenyl)-7-oxo-6-[4-(2-oxopiperidin...,Apixaban,COC1=CC=C(C=C1)N2C3=C(CCN(C3=O)C4=CC=C(C=C4)N5...,"4-[3,5-bis(2-hydroxyphenyl)-1,2,4-triazol-1-yl...",Deferasirox,C1=CC=C(C(=C1)C2=NN(C(=N2)C3=CC=CC=C3O)C4=CC=C...,C1=CC=C(C(=C1)C2=NN(C(=N2)C3=CC=CC=C3O)C4=CC=C...,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1336309,DB14443,ChemicalEntity_ChemicalEntity,104755,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Drugbank,Pubchem,Vibrio cholerae CVD 103-HgR strain live antigen,Vibrio cholerae CVD 103-HgR strain live antigen,NaN,silver(1+),Silver cation,[Ag+],[Ag+],Generalised,Homo sapiens
1336310,108059,ChemicalEntity_ChemicalEntity,DB11074,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Pubchem,Drugbank,"[(8R,9S,10R,13S,14S,17R)-17-acetyl-13-methyl-1...",Segesterone acetate,CC(=O)[C@]1(C(=C)C[C@@H]2[C@@]1(CC[C@H]3[C@H]2...,Dimethicone,Dimethicone,NaN,NaN,Generalised,Homo sapiens
1336311,49831257,ChemicalEntity_ChemicalEntity,442343,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"N-[5-[4-[(1,1-dioxo-1,4-thiazinan-4-yl)methyl]...",Filgotinib,C1CC1C(=O)NC2=NN3C(=N2)C=CC=C3C4=CC=C(C=C4)CN5...,(2S)-6-methyl-2-[(1S)-4-methylcyclohex-3-en-1-...,Levomenol,CC1=CC[C@H](CC1)[C@](C)(CCC=C(C)C)O,CC1=CC[C@H](CC1)[C@](C)(CCC=C(C)C)O,Generalised,Homo sapiens
1336312,44472635,ChemicalEntity_ChemicalEntity,71679,ChemicalEntity,synergistic interaction,ChemicalEntity,PrimeKG,Pubchem,Pubchem,"(6S)-N-[4-[[(2S,5R)-5-[(R)-hydroxy(phenyl)meth...",Vibegron,C1C[C@@H](N[C@@H]1CC2=CC=C(C=C2)NC(=O)[C@@H]3C...,"2-N,2-N,4-N,4-N-tetraethyl-6-hydrazinyl-1,3,5-...",Meladrazine,CCN(CC)C1=NC(=NC(=N1)NN)N(CC)CC,CCN(CC)C1=NC(=NC(=N1)NN)N(CC)CC,Generalised,Homo sapiens


### 2d. PharmKG

In [25]:
PharmKG_Chemical_Chemical = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Chemical_Chemical.csv')
PharmKG_Chemical_Chemical.columns = PharmKG_Chemical_Chemical.columns.str.lower()

PharmKG_Chemical_Chemical['kg_type'] = 'Generalised'
PharmKG_Chemical_Chemical['kg_source'] = 'PharmKG'
PharmKG_Chemical_Chemical['species'] = 'Homo sapiens'

print(f"PharmKG:  {len(PharmKG_Chemical_Chemical):,} rows | columns: {list(PharmKG_Chemical_Chemical.columns)}")
PharmKG_Chemical_Chemical

PharmKG:  33,325 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_orignal', 'tail_orignal', 'pubmed_id', 'sentence_tokenized', 'kg_type', 'species']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_orignal,tail_orignal,pubmed_id,sentence_tokenized,kg_type,species
0,38853,Chemical_Chemical,10261406,chemical,T,chemical,PharmKG,Pubchem,Pubchem,methyldopa,vmf,NaN,NaN,Generalised,Homo sapiens
1,402,Chemical_Chemical,37056,chemical,E-,chemical,PharmKG,Pubchem,Pubchem,hydrogen sulfide,angiotensin converting enzyme,17885553.0,'The novel gaseous vasorelaxant hydrogen_sulfi...,Generalised,Homo sapiens
2,5755,Chemical_Chemical,71587695,chemical,T,chemical,PharmKG,Pubchem,Pubchem,prednisolone,cme,NaN,NaN,Generalised,Homo sapiens
3,11285792,Chemical_Chemical,42625451,chemical,K,chemical,PharmKG,Pubchem,Pubchem,cvc,nos,"24513281.0, 24446550.0","'Accordingly , these data suggest , in part , ...",Generalised,Homo sapiens
4,44356648,Chemical_Chemical,977,chemical,Q,chemical,PharmKG,Pubchem,Pubchem,tnfalpha,lox,"17673218.0, 17673218.0, 17673218.0, 19406911.0...",'Our hypothesis is that the inflammatory cytok...,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33320,2370,Chemical_Chemical,7099,chemical,E,chemical,PharmKG,Pubchem,Pubchem,bethanechol,prl,NaN,NaN,Generalised,Homo sapiens
33321,99881,Chemical_Chemical,76288,chemical,Pr,chemical,PharmKG,Pubchem,Pubchem,dmbpo,hep,23961165.0,'DMBPO exhibited cytotoxic activity on HEP 2 a...,Generalised,Homo sapiens
33322,5460341,Chemical_Chemical,446257,chemical,J,chemical,PharmKG,Pubchem,Pubchem,calcium,iri,"15924816.0, 22717637.0, 15924816.0, 22717637.0...","'-LRB- 5 -RRB- In sham-operated group , the bi...",Generalised,Homo sapiens
33323,5743,Chemical_Chemical,9860606,chemical,T,chemical,PharmKG,Pubchem,Pubchem,dexamethasone,eh,"8579139.0, 4090833.0","'Dexamethasone , diuretics , and a room air re...",Generalised,Homo sapiens


### 2e. Hetionet

In [27]:
Hetionet_Chemical_Chemical = pd.read_csv(PROC_DIR + 'Hetionet/Compound_Compound_Hetionet.csv')
Hetionet_Chemical_Chemical.columns = Hetionet_Chemical_Chemical.columns.str.lower()
print(f"Hetionet: {len(Hetionet_Chemical_Chemical):,} rows | columns: {list(Hetionet_Chemical_Chemical.columns)}")

Hetionet_Chemical_Chemical['kg_type'] = 'Generalised'
Hetionet_Chemical_Chemical['kg_source'] = 'Hetionet'
Hetionet_Chemical_Chemical['species'] = 'Homo sapiens'

Hetionet_Chemical_Chemical

Hetionet: 6,486 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_pubchem_mapped', 'tail_pubchem_mapped']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_pubchem_mapped,tail_pubchem_mapped,kg_type,species
0,4927,Compound_Compound,4940,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB01069,DB00777,Generalised,Homo sapiens
1,237,Compound_Compound,2165,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB01103,DB00613,Generalised,Homo sapiens
2,3055,Compound_Compound,5284632,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB01231,DB00209,Generalised,Homo sapiens
3,33255,Compound_Compound,5479529,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB01327,DB01112,Generalised,Homo sapiens
4,2812,Compound_Compound,2378,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB00257,DB04794,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6481,5340,Compound_Compound,9047,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB06147,DB00664,Generalised,Homo sapiens
6482,4057,Compound_Compound,3494,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB04843,DB00986,Generalised,Homo sapiens
6483,9878,Compound_Compound,31307,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB00324,DB00620,Generalised,Homo sapiens
6484,3241,Compound_Compound,34312,Compound,Compound–resembles–Compound,Compound,Hetionet,Pubchem,Pubchem,DB00751,DB00776,Generalised,Homo sapiens


### 2f. CrossBAR

In [29]:
CrossBAR_Chemical_Chemical = pd.read_csv(PROC_DIR + 'crossbar/ChemicalEntity_ChemicalEntity_Crossbar.csv')
CrossBAR_Chemical_Chemical.columns = CrossBAR_Chemical_Chemical.columns.str.lower()
print(f"CrossBAR: {len(CrossBAR_Chemical_Chemical):,} rows | columns: {list(CrossBAR_Chemical_Chemical.columns)}")
CrossBAR_Chemical_Chemical['kg_type'] = 'Generalised'
CrossBAR_Chemical_Chemical['kg_source'] = 'crossbar'
CrossBAR_Chemical_Chemical['species'] = 'Homo sapiens'

CrossBAR_Chemical_Chemical

CrossBAR: 50,019 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'head_detail_name', 'tail_alt_id', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_alt_id,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,46173812,Chemical_Chemical,46931119,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:53028,4-deoxy-4-formamido-alpha-L-arabinopyranosyl d...,CHEBI:58909,4-deoxy-4-formamido-alpha-L-arabinopyranosyl d...,Pubchem,Pubchem,Generalised,Homo sapiens
1,5280917,Chemical_Chemical,45266667,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:27513,precorrin-6X,CHEBI:58513,precorrin-6X(8-),Pubchem,Pubchem,Generalised,Homo sapiens
2,10483388,Chemical_Chemical,11711453,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:66606,(+)-lyoniresinol-3-alpha-O-beta-D-glucopyranoside,CHEBI:68168,(+)-lyoniresinol,Pubchem,Pubchem,Generalised,Homo sapiens
3,10313435,Chemical_Chemical,44237242,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:180670,4-methoxy-TEMPO,CHEBI:26151,piperidines,Pubchem,Pubchem,Generalised,Homo sapiens
4,90657799,Chemical_Chemical,6439171,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:90889,aurachin C epoxide,CHEBI:90888,aurachin C,Pubchem,Pubchem,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50014,170336,Chemical_Chemical,667500,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:59329,(S)-carbinoxamine,CHEBI:59328,(R)-carbinoxamine,Pubchem,Pubchem,Generalised,Homo sapiens
50015,439697,Chemical_Chemical,4633873,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:50447,(S)-carnitinamide,CHEBI:48604,carnitinamide,Pubchem,Pubchem,Generalised,Homo sapiens
50016,21604111,Chemical_Chemical,92825,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:66157,salzmannianoside A,CHEBI:5580,gypsogenin,Pubchem,Pubchem,Generalised,Homo sapiens
50017,1150567,Chemical_Chemical,6971138,ChemicalEntity,ChemicalEntity,crossbar,CHEBI:53292,(R)-donepezil,CHEBI:145502,(R)-donepezil(1+),Pubchem,Pubchem,Generalised,Homo sapiens


### 2g. iBKH

In [31]:
iBKH_Chemical_Chemical = pd.read_csv(PROC_DIR + 'iBKH/Chemical_Chemical_ibkh.csv')
iBKH_Chemical_Chemical.columns = iBKH_Chemical_Chemical.columns.str.lower()
print(f"iBKH:     {len(iBKH_Chemical_Chemical):,} rows | columns: {list(iBKH_Chemical_Chemical.columns)}")

iBKH_Chemical_Chemical['kg_type'] = 'Generalised'
iBKH_Chemical_Chemical['kg_source'] = 'ibkh'
iBKH_Chemical_Chemical['species'] = 'Homo sapiens'

iBKH_Chemical_Chemical

iBKH:     2,684,682 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


/tmp/ipykernel_2687068/4159091144.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  iBKH_Chemical_Chemical = pd.read_csv(PROC_DIR + 'iBKH/Chemical_Chemical_ibkh.csv')


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,DB00001,ChemicalEntity_ChemicalEntity,10182969,ChemicalEntity,ChemicalEntity,ibkh,Lepirudin,NaN,1-(4-methoxyphenyl)-7-oxo-6-[4-(2-oxopiperidin...,Drugbank,Pubchem,Generalised,Homo sapiens
1,DB00001,ChemicalEntity_ChemicalEntity,135565674,ChemicalEntity,ChemicalEntity,ibkh,Lepirudin,NaN,ethyl 3-[[2-[[4-(N-hexoxycarbonylcarbamimidoyl...,Drugbank,Pubchem,Generalised,Homo sapiens
2,DB00001,ChemicalEntity_ChemicalEntity,3062316,ChemicalEntity,ChemicalEntity,ibkh,Lepirudin,NaN,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,Drugbank,Pubchem,Generalised,Homo sapiens
3,DB00001,ChemicalEntity_ChemicalEntity,214348,ChemicalEntity,ChemicalEntity,ibkh,Lepirudin,NaN,"4-[3,5-bis(2-hydroxyphenyl)-1,2,4-triazol-1-yl...",Drugbank,Pubchem,Generalised,Homo sapiens
4,DB00001,ChemicalEntity_ChemicalEntity,31401,ChemicalEntity,ChemicalEntity,ibkh,Lepirudin,NaN,"(4R)-4-[(3R,5S,7S,8R,9S,10S,13R,14S,17R)-3,7-d...",Drugbank,Pubchem,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2684677,34359,ChemicalEntity_ChemicalEntity,3779,ChemicalEntity,ChemicalEntity,ibkh,"(2S)-3-(3,4-dihydroxyphenyl)-2-hydrazinyl-2-me...",C[C@](CC1=CC(=C(C=C1)O)O)(C(=O)O)NN,4-[1-hydroxy-2-(propan-2-ylamino)ethyl]benzene...,Pubchem,Pubchem,Generalised,Homo sapiens
2684678,51081,ChemicalEntity_ChemicalEntity,10178705,ChemicalEntity,ChemicalEntity,ibkh,1-ethyl-6-fluoro-7-(4-methylpiperazin-1-yl)-4-...,CCN1C=C(C(=O)C2=CC(=C(C=C21)N3CCN(CC3)C)F)C(=O)O,7-[(3R)-3-aminoazepan-1-yl]-8-chloro-1-cyclopr...,Pubchem,Pubchem,Generalised,Homo sapiens
2684679,135403821,ChemicalEntity_ChemicalEntity,6436173,ChemicalEntity,ChemicalEntity,ibkh,"[(7S,9E,11S,12R,13S,14R,15R,16R,17S,18S,19E,21...",C[C@H]1/C=C/C=C(\C(=O)NC2=C(C(=C3C(=C2O)C(=C(C...,"[(7S,9E,11S,12R,13S,14R,15R,16R,17S,18S,19E,21...",Pubchem,Pubchem,Generalised,Homo sapiens
2684680,5340,ChemicalEntity_ChemicalEntity,9047,ChemicalEntity,ChemicalEntity,ibkh,"4-amino-N-(1,3-thiazol-2-yl)benzenesulfonamide",C1=CC(=CC=C1N)S(=O)(=O)NC2=NC=CS2,4-amino-N-(3-methoxypyrazin-2-yl)benzenesulfon...,Pubchem,Pubchem,Generalised,Homo sapiens


### 2h. DtiNet

In [33]:
DtiNet_Chemical_Chemical = pd.read_csv(PROC_DIR + 'DTINet/Chemical_Chemical_DTINet.csv')
DtiNet_Chemical_Chemical.columns = DtiNet_Chemical_Chemical.columns.str.lower()
print(f"DtiNet:   {len(DtiNet_Chemical_Chemical):,} rows | columns: {list(DtiNet_Chemical_Chemical.columns)}")

DtiNet_Chemical_Chemical['kg_type'] = 'Generalised'
DtiNet_Chemical_Chemical['kg_source'] = 'dtinet'
DtiNet_Chemical_Chemical['species'] = 'Homo sapiens'

DtiNet_Chemical_Chemical

DtiNet:   10,035 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'relation_source']


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,relation_source,kg_type,species
0,445354,ChemicalEntity_ChemicalEntity,82146,ChemicalEntity,ChemicalEntity,dtinet,DB00162,"(2E,4E,6E,8E)-3,7-dimethyl-9-(2,6,6-trimethylc...",CC1=C(C(CCC1)(C)C)/C=C/C(=C/C=C/C(=C/CO)/C)/C,"4-[1-(3,5,5,8,8-pentamethyl-6,7-dihydronaphtha...",Pubchem,Pubchem,DTINet,Generalised,Homo sapiens
1,445354,ChemicalEntity_ChemicalEntity,5284513,ChemicalEntity,ChemicalEntity,dtinet,DB00162,"(2E,4E,6E,8E)-3,7-dimethyl-9-(2,6,6-trimethylc...",CC1=C(C(CCC1)(C)C)/C=C/C(=C/C=C/C(=C/CO)/C)/C,"(2E,4E,6E,8E)-9-(4-methoxy-2,3,6-trimethylphen...",Pubchem,Pubchem,DTINet,Generalised,Homo sapiens
2,445354,ChemicalEntity_ChemicalEntity,3034010,ChemicalEntity,ChemicalEntity,dtinet,DB00162,"(2E,4E,6E,8E)-3,7-dimethyl-9-(2,6,6-trimethylc...",CC1=C(C(CCC1)(C)C)/C=C/C(=C/C=C/C(=C/CO)/C)/C,"[(2S)-1-[(2S,3S)-3-hexyl-4-oxooxetan-2-yl]trid...",Pubchem,Pubchem,DTINet,Generalised,Homo sapiens
3,54687,ChemicalEntity_ChemicalEntity,54682461,ChemicalEntity,ChemicalEntity,dtinet,DB00175,"(3R,5R)-7-[(1S,2S,6S,8S,8aR)-6-hydroxy-2-methy...",CC[C@H](C)C(=O)O[C@H]1C[C@@H](C=C2[C@H]1[C@H](...,N-[3-[(1R)-1-[(2R)-4-hydroxy-6-oxo-2-(2-phenyl...,Pubchem,Pubchem,DTINet,Generalised,Homo sapiens
4,54687,ChemicalEntity_ChemicalEntity,3339,ChemicalEntity,ChemicalEntity,dtinet,DB00175,"(3R,5R)-7-[(1S,2S,6S,8S,8aR)-6-hydroxy-2-methy...",CC[C@H](C)C(=O)O[C@H]1C[C@@H](C=C2[C@H]1[C@H](...,propan-2-yl 2-[4-(4-chlorobenzoyl)phenoxy]-2-m...,Pubchem,Pubchem,DTINet,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10030,5282452,ChemicalEntity_ChemicalEntity,12560,ChemicalEntity,ChemicalEntity,dtinet,DB08860,"(E,3R,5S)-7-[2-cyclopropyl-4-(4-fluorophenyl)q...",C1CC1C2=NC3=CC=CC=C3C(=C2/C=C/[C@H](C[C@H](CC(...,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Pubchem,Pubchem,DTINet,Generalised,Homo sapiens
10031,5282452,ChemicalEntity_ChemicalEntity,3339,ChemicalEntity,ChemicalEntity,dtinet,DB08860,"(E,3R,5S)-7-[2-cyclopropyl-4-(4-fluorophenyl)q...",C1CC1C2=NC3=CC=CC=C3C(=C2/C=C/[C@H](C[C@H](CC(...,propan-2-yl 2-[4-(4-chlorobenzoyl)phenoxy]-2-m...,Pubchem,Pubchem,DTINet,Generalised,Homo sapiens
10032,5282452,ChemicalEntity_ChemicalEntity,148192,ChemicalEntity,ChemicalEntity,dtinet,DB08860,"(E,3R,5S)-7-[2-cyclopropyl-4-(4-fluorophenyl)q...",C1CC1C2=NC3=CC=CC=C3C(=C2/C=C/[C@H](C[C@H](CC(...,"methyl N-[(2S)-1-[2-[(2S,3S)-2-hydroxy-3-[[(2S...",Pubchem,Pubchem,DTINet,Generalised,Homo sapiens
10033,5282452,ChemicalEntity_ChemicalEntity,3463,ChemicalEntity,ChemicalEntity,dtinet,DB08860,"(E,3R,5S)-7-[2-cyclopropyl-4-(4-fluorophenyl)q...",C1CC1C2=NC3=CC=CC=C3C(=C2/C=C/[C@H](C[C@H](CC(...,"5-(2,5-dimethylphenoxy)-2,2-dimethylpentanoic ...",Pubchem,Pubchem,DTINet,Generalised,Homo sapiens


### 2i. STITCH

In [35]:
STITCH_Chemical_Chemical = pd.read_csv(PROC_DIR + 'stitch/stitch_HUMAN_Chemical_Chemical_STITCH.csv')
STITCH_Chemical_Chemical.columns = STITCH_Chemical_Chemical.columns.str.lower()
print(f"STITCH:   {len(STITCH_Chemical_Chemical):,} rows | columns: {list(STITCH_Chemical_Chemical.columns)}")

STITCH_Chemical_Chemical['kg_type'] = 'Generalised'
STITCH_Chemical_Chemical['kg_source'] = 'stitch'
STITCH_Chemical_Chemical['species'] = 'Homo sapiens'

STITCH_Chemical_Chemical

STITCH:   716,016 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smile_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,91392180,ChemicalEntity_ChemicalEntity,10071196,ChemicalEntity,ChemicalEntity,stitch,"6,6-dimethyl-11-prop-2-ynoxy-5,6,6a,7-tetrahyd...",C[N+]1(CCC2=C3C1CC4=C(C3=CC=C2)C(=CC=C4)OCC#C)C,1-[(4-fluorophenyl)methyl]-1-(1-methylpiperidi...,Pubchem,Pubchem,Generalised,Homo sapiens
1,91392180,ChemicalEntity_ChemicalEntity,107992,ChemicalEntity,ChemicalEntity,stitch,"6,6-dimethyl-11-prop-2-ynoxy-5,6,6a,7-tetrahyd...",C[N+]1(CCC2=C3C1CC4=C(C3=CC=C2)C(=CC=C4)OCC#C)C,2-chloro-6-piperazin-1-ylpyrazine,Pubchem,Pubchem,Generalised,Homo sapiens
2,91392180,ChemicalEntity_ChemicalEntity,11237985,ChemicalEntity,ChemicalEntity,stitch,"6,6-dimethyl-11-prop-2-ynoxy-5,6,6a,7-tetrahyd...",C[N+]1(CCC2=C3C1CC4=C(C3=CC=C2)C(=CC=C4)OCC#C)C,"[(3R)-1-azabicyclo[2.2.2]octan-3-yl] 6-[(3S,4R...",Pubchem,Pubchem,Generalised,Homo sapiens
3,91392180,ChemicalEntity_ChemicalEntity,11430856,ChemicalEntity,ChemicalEntity,stitch,"6,6-dimethyl-11-prop-2-ynoxy-5,6,6a,7-tetrahyd...",C[N+]1(CCC2=C3C1CC4=C(C3=CC=C2)C(=CC=C4)OCC#C)C,N-[3-[4-[4-(cyclohexylmethylsulfonylamino)buty...,Pubchem,Pubchem,Generalised,Homo sapiens
4,91392180,ChemicalEntity_ChemicalEntity,115292,ChemicalEntity,ChemicalEntity,stitch,"6,6-dimethyl-11-prop-2-ynoxy-5,6,6a,7-tetrahyd...",C[N+]1(CCC2=C3C1CC4=C(C3=CC=C2)C(=CC=C4)OCC#C)C,3-(1-aminopropan-2-yl)-1H-indol-5-ol,Pubchem,Pubchem,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
716011,1,ChemicalEntity_ChemicalEntity,9833257,ChemicalEntity,ChemicalEntity,stitch,3-acetyloxy-4-(trimethylazaniumyl)butanoate,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,(3S)-3-amino-4-[[(2S)-1-[[(2S)-1-[[2-[[(2S)-1-...,Pubchem,Pubchem,Generalised,Homo sapiens
716012,1,ChemicalEntity_ChemicalEntity,11020241,ChemicalEntity,ChemicalEntity,stitch,3-acetyloxy-4-(trimethylazaniumyl)butanoate,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,"calcium;(2R)-2-[(1S)-1,2-dihydroxyethyl]-5-oxo...",Pubchem,Pubchem,Generalised,Homo sapiens
716013,1,ChemicalEntity_ChemicalEntity,50878598,ChemicalEntity,ChemicalEntity,stitch,3-acetyloxy-4-(trimethylazaniumyl)butanoate,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,"(3Z)-3-[(2E)-2-[(1R,7aR)-7a-methyl-1-[(2R)-6-m...",Pubchem,Pubchem,Generalised,Homo sapiens
716014,1,ChemicalEntity_ChemicalEntity,71300633,ChemicalEntity,ChemicalEntity,stitch,3-acetyloxy-4-(trimethylazaniumyl)butanoate,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,2-(butylamino)propan-2-ylphosphonic acid;cobal...,Pubchem,Pubchem,Generalised,Homo sapiens


### 2j. Phenknowlator

In [37]:
pheknowlator = pd.read_csv(PROC_DIR + 'pheknowlator/Chemical_Chemical.csv')
pheknowlator = pheknowlator.rename(columns={'relation_source': 'kg_source'})
pheknowlator.columns = pheknowlator.columns.str.lower()
# Drop null heads; remove float-cast artefacts from integer CIDs
pheknowlator = pheknowlator[~pheknowlator['head'].isna()]
pheknowlator['head'] = pheknowlator['head'].astype(str).str.replace(r'\.0$', '', regex=True)
print(f"pheknowlator:   {len(pheknowlator):,} rows | columns: {list(pheknowlator.columns)}")

pheknowlator['relation'] = 'ChemicalEntity_ChemicalEntity'
pheknowlator['head_type'] = 'ChemicalEntity'
pheknowlator['tail_type'] = 'ChemicalEntity'

pheknowlator['kg_type'] = 'Generalised'
pheknowlator['kg_source'] = 'pheknowlator'
pheknowlator['species'] = 'Homo sapiens'

pheknowlator

pheknowlator:   14,698 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'head_iupac', 'tail_iupac', 'head_smiles', 'tail_smiles', 'head_id_is', 'tail_id_is', 'kg_source']


,head,relation,tail,head_type,tail_type,head_iupac,tail_iupac,head_smiles,tail_smiles,head_id_is,tail_id_is,kg_source,kg_type,species
0,2805684,ChemicalEntity_ChemicalEntity,10176084,ChemicalEntity,ChemicalEntity,"3-[1-(2,4-dichlorophenyl)sulfonylpyrrolidin-2-...","1,2,3,4-tetrahydropyridine",C1CC(N(C1)S(=O)(=O)C2=C(C=C(C=C2)Cl)Cl)C3=CN=C...,C1CC=CNC1,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens
1,122164832,ChemicalEntity_ChemicalEntity,445638,ChemicalEntity,ChemicalEntity,[(2R)-2-[(Z)-hexadec-9-enoyl]oxy-3-hydroxyprop...,(Z)-hexadec-9-enoic acid,CCCCCC/C=C\CCCCCCCC(=O)O[C@H](CO)COP(=O)([O-])...,CCCCCC/C=C\CCCCCCCC(=O)O,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens
2,3083575,ChemicalEntity_ChemicalEntity,3083575,ChemicalEntity,ChemicalEntity,"2,8-dihydroxy-1-methoxy-3-methylanthracene-9,1...","2,8-dihydroxy-1-methoxy-3-methylanthracene-9,1...",CC1=CC2=C(C(=C1O)OC)C(=O)C3=C(C2=O)C=CC=C3O,CC1=CC2=C(C(=C1O)OC)C(=O)C3=C(C2=O)C=CC=C3O,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens
3,60184949,ChemicalEntity_ChemicalEntity,44237242,ChemicalEntity,ChemicalEntity,"(1S,5R)-3-(2-fluorophenyl)sulfonyl-7-[4-(3-met...","2-(hydroxymethyl)piperidin-1-ium-3,4,5-triol",COC1=CC=CC(=C1)C2=CC=C(C=C2)C3[C@H]4CN(C[C@@H]...,C1C(C(C(C([NH2+]1)CO)O)O)O,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens
4,50986185,ChemicalEntity_ChemicalEntity,139087999,ChemicalEntity,ChemicalEntity,"(1S,4R,4aS)-1,6-dimethyl-4-propan-2-yl-1,2,3,4...","1-[(1R,3aS,6R,7R,7aS)-6-hydroxy-4-methylidene-...",C[C@H]1CC[C@@H]([C@@H]2C1=CCC(=C2)C)C(C)C,CC(C)[C@H]1[C@@H](CC(=C)[C@@H]2[C@@H]1[C@@H](C...,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14693,11914,ChemicalEntity_ChemicalEntity,1292,ChemicalEntity,ChemicalEntity,(2R)-2-hydroxy-2-phenylacetic acid,2-hydroxy-2-phenylacetic acid,C1=CC=C(C=C1)[C@H](C(=O)O)O,C1=CC=C(C=C1)C(C(=O)O)O,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens
14694,6018498,ChemicalEntity_ChemicalEntity,44237242,ChemicalEntity,ChemicalEntity,[4-[(Z)-(thiophene-2-carbonylhydrazinylidene)m...,"2-(hydroxymethyl)piperidin-1-ium-3,4,5-triol",CC1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)/C=N\NC(=O)C3...,C1C(C(C(C([NH2+]1)CO)O)O)O,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens
14695,10241693,ChemicalEntity_ChemicalEntity,635,ChemicalEntity,ChemicalEntity,4-[[tert-butyl(hydroxy)amino]methylidene]cyclo...,"(3,5-dihydroxyoxolan-2-yl)methyl dihydrogen ph...",CC(C)(C)N(C=C1C=CC(=O)C=C1)O,C1C(C(OC1O)COP(=O)(O)O)O,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens
14696,440881,ChemicalEntity_ChemicalEntity,16069985,ChemicalEntity,ChemicalEntity,"[(2R,3R,4S,5S,6R)-5-acetamido-3,4-dihydroxy-6-...","[(3R,4S,5S,6R)-5-acetamido-3,4-dihydroxy-6-met...",C[C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)OP(=O)(O)O...,C[C@@H]1[C@H]([C@@H]([C@H](C(O1)OP(=O)(O)OP(=O...,Pubchem,Pubchem,pheknowlator,Generalised,Homo sapiens


### 2k. HALD

In [45]:
hald = pd.read_csv(PROC_DIR + 'hald/Chemical_Chemical_HALD.csv')
hald = hald.rename(columns={'relation_source': 'kg_source'})
hald.columns = hald.columns.str.lower()
# Drop null heads; remove float-cast artefacts from integer CIDs
hald = hald[~hald['head'].isna()]
hald['head'] = hald['head'].astype(str).str.replace(r'\.0$', '', regex=True)
print(f"hald:   {len(hald):,} rows | columns: {list(hald.columns)}")

hald['kg_type'] = 'Aging'
hald['kg_source'] = 'hald'
hald['species'] = 'Homo sapiens'
hald

hald:   1,045 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'tail_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_smiles_name,tail_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,24778708,ChemicalEntity_ChemicalEntity,Fatty Acids,ChemicalEntity,contain,ChemicalEntity,hald,(3-hexadecanoyloxy-2-nonadecanoyloxypropyl) 2-...,CCCCCCCCCCCCCCCCCCC(=O)OC(COC(=O)CCCCCCCCCCCCC...,Fatty Acids,Fatty Acids,Pubchem,NaN,Aging,Homo sapiens
1,24778708,ChemicalEntity_ChemicalEntity,Monounsaturated,ChemicalEntity,contain,ChemicalEntity,hald,(3-hexadecanoyloxy-2-nonadecanoyloxypropyl) 2-...,CCCCCCCCCCCCCCCCCCC(=O)OC(COC(=O)CCCCCCCCCCCCC...,Monounsaturated,Monounsaturated,Pubchem,NaN,Aging,Homo sapiens
2,Ceramides,ChemicalEntity_ChemicalEntity,5793,ChemicalEntity,reduce,ChemicalEntity,hald,Ceramides,Ceramides,C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",NaN,Pubchem,Aging,Homo sapiens
3,637517,ChemicalEntity_ChemicalEntity,Fatty Acids,ChemicalEntity,comprise,ChemicalEntity,hald,(E)-octadec-9-enoic acid,CCCCCCCC/C=C/CCCCCCCC(=O)O,Fatty Acids,Fatty Acids,Pubchem,NaN,Aging,Homo sapiens
4,24778708,ChemicalEntity_ChemicalEntity,Phosphatidylethanolamines,ChemicalEntity,contain,ChemicalEntity,hald,(3-hexadecanoyloxy-2-nonadecanoyloxypropyl) 2-...,CCCCCCCCCCCCCCCCCCC(=O)OC(COC(=O)CCCCCCCCCCCCC...,Phosphatidylethanolamines,Phosphatidylethanolamines,Pubchem,NaN,Aging,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1040,5282411,ChemicalEntity_ChemicalEntity,DB01109,ChemicalEntity,confer,ChemicalEntity,hald,"(5Z)-5-[(3aR,4R,5R,6aS)-5-hydroxy-4-[(E,3S)-3-...",CCCCC[C@@H](/C=C/[C@H]1[C@@H](C[C@H]2[C@@H]1C/...,Heparin,Heparin,Pubchem,Drugbank,Aging,Homo sapiens
1041,448601,ChemicalEntity_ChemicalEntity,5793,ChemicalEntity,cause,ChemicalEntity,hald,"(4R,7S,10S,13R,16S,19R)-10-(4-aminobutyl)-19-[...",C[C@H]([C@H]1C(=O)N[C@@H](CSSC[C@@H](C(=O)N[C@...,C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",Pubchem,Pubchem,Aging,Homo sapiens
1042,134601,ChemicalEntity_ChemicalEntity,Lipofuscin,ChemicalEntity,decrease,ChemicalEntity,hald,(3S)-3-amino-4-[[(2S)-1-methoxy-1-oxo-3-phenyl...,COC(=O)[C@H](CC1=CC=CC=C1)NC(=O)[C@H](CC(=O)O)N,Lipofuscin,Lipofuscin,Pubchem,NaN,Aging,Homo sapiens
1043,Fatty Acids,ChemicalEntity_ChemicalEntity,124886,ChemicalEntity,affect,ChemicalEntity,hald,Fatty Acids,Fatty Acids,C(CC(=O)N[C@@H](CS)C(=O)NCC(=O)O)[C@@H](C(=O)O)N,(2S)-2-amino-5-[[(2R)-1-(carboxymethylamino)-1...,NaN,Pubchem,Aging,Homo sapiens


### 2l. AgeAnnoMO

In [46]:
AgeAnnoMO = pd.read_csv(PROC_DIR + 'ageanno_processed_mo/AgeAnnoMO_Human_Cemical_Chemical_1.csv')
AgeAnnoMO.columns = AgeAnnoMO.columns.str.lower()

AgeAnnoMO['kg_type'] = 'Aging'
AgeAnnoMO['kg_source'] = 'ageannomo'
AgeAnnoMO['species'] = 'Homo sapiens'
AgeAnnoMO

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source,species,head_id,head_id_is,tail_id,tail_id_is,kg_type,kg_source
0,69726,ChemicalEntity_ChemicalEntity,80220,ChemicalEntity,ChemicalEntity,"1-methyl-7,9-dihydro-3H-purine-2,6,8-trione","1-methyl-3,7-dihydropurine-2,6-dione",Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,1-Methyluric acid,Pubchem,1-Methylxanthine,Pubchem,Aging,ageannomo
1,69726,ChemicalEntity_ChemicalEntity,700653,ChemicalEntity,ChemicalEntity,"1-methyl-7,9-dihydro-3H-purine-2,6,8-trione",(2S)-2-acetamido-3-(1H-indol-3-yl)propanoic acid,Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,1-Methyluric acid,Pubchem,N-Acetyl-L-tryptophan,Pubchem,Aging,ageannomo
2,80220,ChemicalEntity_ChemicalEntity,700653,ChemicalEntity,ChemicalEntity,"1-methyl-3,7-dihydropurine-2,6-dione",(2S)-2-acetamido-3-(1H-indol-3-yl)propanoic acid,Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,1-Methylxanthine,Pubchem,N-Acetyl-L-tryptophan,Pubchem,Aging,ageannomo
3,80220,ChemicalEntity_ChemicalEntity,107461,ChemicalEntity,ChemicalEntity,"1-methyl-3,7-dihydropurine-2,6-dione","N-[1-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxyme...",Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,1-Methylxanthine,Pubchem,N4-Acetylcytidine,Pubchem,Aging,ageannomo
4,107461,ChemicalEntity_ChemicalEntity,108192,ChemicalEntity,ChemicalEntity,"N-[1-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxyme...","(2S,3S,4S,5R,6R)-6-[[(8R,9S,10R,13S,14S,17S)-1...",Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,N4-Acetylcytidine,Pubchem,Testosterone glucuronide,Pubchem,Aging,ageannomo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
637,154035,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,ChemicalEntity,"(2S,3S,4S,5R,6S)-3,4,5-trihydroxy-6-(4-methylp...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,p-Cresol glucuronide,Pubchem,Chenodeoxycholic acid 3-sulfate,Pubchem,Aging,ageannomo
638,11902892,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,ChemicalEntity,"(2S,4R)-4-hydroxy-1-[(2S)-pyrrolidin-1-ium-2-c...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,"(2S,4R)-4-hydroxy-1-[(2S)-pyrrolidin-1-ium-2-c...",Pubchem,Chenodeoxycholic acid 3-sulfate,Pubchem,Aging,ageannomo
639,9085,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,ChemicalEntity,(2S)-2-amino-6-(diaminomethylideneamino)hexano...,"(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,L-Homoarginine,Pubchem,Chenodeoxycholic acid 3-sulfate,Pubchem,Aging,ageannomo
640,88299,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,ChemicalEntity,"N-(6-amino-3-methyl-2,4-dioxo-1H-pyrimidin-5-y...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Pubchem,Pubchem,AgeAnnoMO,Homo sapiens,5-Acetylamino-6-amino-3-methyluracil,Pubchem,Chenodeoxycholic acid 3-sulfate,Pubchem,Aging,ageannomo


In [50]:
# ── Step 1: Inspect duplicate columns in each DataFrame ──────────────────────
for name, df in [
    ('Monarch_Chemical_Chemical',   Monarch_Chemical_Chemical),
    ('DRKG_Chemical_Chemical',      DRKG_Chemical_Chemical),
    ('PrimeKG_Chemical_Chemical_1', PrimeKG_Chemical_Chemical_1),
    ('PrimeKG_Chemical_Chemical_2', PrimeKG_Chemical_Chemical_2),
    ('PharmKG_Chemical_Chemical',   PharmKG_Chemical_Chemical),
    ('Hetionet_Chemical_Chemical',  Hetionet_Chemical_Chemical),
    ('CrossBAR_Chemical_Chemical',  CrossBAR_Chemical_Chemical),
    ('iBKH_Chemical_Chemical',      iBKH_Chemical_Chemical),
    ('DtiNet_Chemical_Chemical',    DtiNet_Chemical_Chemical),
    ('STITCH_Chemical_Chemical',    STITCH_Chemical_Chemical),
    ('pheknowlator',                pheknowlator),
    ('hald',                        hald),
    ('AgeAnnoMO',                   AgeAnnoMO)
]:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            print(f"  '{col}':")
            for i, (_, series) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"    occurrence {i} → sample values: {series.dropna().head(3).tolist()}")


[AgeAnnoMO] duplicate columns: ['head_id_is', 'tail_id_is']
  'head_id_is':
    occurrence 0 → sample values: ['Pubchem', 'Pubchem', 'Pubchem']
    occurrence 1 → sample values: ['Pubchem', 'Pubchem', 'Pubchem']
  'tail_id_is':
    occurrence 0 → sample values: ['Pubchem', 'Pubchem', 'Pubchem']
    occurrence 1 → sample values: ['Pubchem', 'Pubchem', 'Pubchem']


In [51]:
# ── Step 2: Drop specific occurrence (keep='first' or keep='last') ────────────
# Edit per-DataFrame based on what you saw above

# keep='first' → keeps occurrence 0, drops occurrence 1
# keep='last'  → keeps occurrence 1, drops occurrence 0

Monarch_Chemical_Chemical   = Monarch_Chemical_Chemical.loc[:,   ~Monarch_Chemical_Chemical.columns.duplicated(keep='first')]
DRKG_Chemical_Chemical      = DRKG_Chemical_Chemical.loc[:,      ~DRKG_Chemical_Chemical.columns.duplicated(keep='first')]
PrimeKG_Chemical_Chemical_1 = PrimeKG_Chemical_Chemical_1.loc[:, ~PrimeKG_Chemical_Chemical_1.columns.duplicated(keep='first')]
PrimeKG_Chemical_Chemical_2 = PrimeKG_Chemical_Chemical_2.loc[:, ~PrimeKG_Chemical_Chemical_2.columns.duplicated(keep='first')]
PharmKG_Chemical_Chemical   = PharmKG_Chemical_Chemical.loc[:,   ~PharmKG_Chemical_Chemical.columns.duplicated(keep='first')]
Hetionet_Chemical_Chemical  = Hetionet_Chemical_Chemical.loc[:,  ~Hetionet_Chemical_Chemical.columns.duplicated(keep='first')]
CrossBAR_Chemical_Chemical  = CrossBAR_Chemical_Chemical.loc[:,  ~CrossBAR_Chemical_Chemical.columns.duplicated(keep='first')]
iBKH_Chemical_Chemical      = iBKH_Chemical_Chemical.loc[:,      ~iBKH_Chemical_Chemical.columns.duplicated(keep='first')]
DtiNet_Chemical_Chemical    = DtiNet_Chemical_Chemical.loc[:,    ~DtiNet_Chemical_Chemical.columns.duplicated(keep='first')]
STITCH_Chemical_Chemical    = STITCH_Chemical_Chemical.loc[:,    ~STITCH_Chemical_Chemical.columns.duplicated(keep='first')]
pheknowlator                = pheknowlator.loc[:,                ~pheknowlator.columns.duplicated(keep='first')]
hald                        = hald.loc[:,                        ~hald.columns.duplicated(keep='first')]
AgeAnnoMO                   = AgeAnnoMO.loc[:,                   ~AgeAnnoMO.columns.duplicated(keep='first')]

# ── Verify all clean ──────────────────────────────────────────────────────────
for name, df in [
    ('Monarch_Chemical_Chemical',   Monarch_Chemical_Chemical),
    ('DRKG_Chemical_Chemical',      DRKG_Chemical_Chemical),
    ('PrimeKG_Chemical_Chemical_1', PrimeKG_Chemical_Chemical_1),
    ('PrimeKG_Chemical_Chemical_2', PrimeKG_Chemical_Chemical_2),
    ('PharmKG_Chemical_Chemical',   PharmKG_Chemical_Chemical),
    ('Hetionet_Chemical_Chemical',  Hetionet_Chemical_Chemical),
    ('CrossBAR_Chemical_Chemical',  CrossBAR_Chemical_Chemical),
    ('iBKH_Chemical_Chemical',      iBKH_Chemical_Chemical),
    ('DtiNet_Chemical_Chemical',    DtiNet_Chemical_Chemical),
    ('STITCH_Chemical_Chemical',    STITCH_Chemical_Chemical),
    ('pheknowlator',                pheknowlator),
    ('hald',                        hald),
    ('AgeAnnoMO',                   AgeAnnoMO)
]:
    remaining = df.columns[df.columns.duplicated()].tolist()
    status = f"✓ clean" if not remaining else f"✗ still has dupes: {remaining}"
    print(f"[{name}] {status}")

[Monarch_Chemical_Chemical] ✓ clean
[DRKG_Chemical_Chemical] ✓ clean
[PrimeKG_Chemical_Chemical_1] ✓ clean
[PrimeKG_Chemical_Chemical_2] ✓ clean
[PharmKG_Chemical_Chemical] ✓ clean
[Hetionet_Chemical_Chemical] ✓ clean
[CrossBAR_Chemical_Chemical] ✓ clean
[iBKH_Chemical_Chemical] ✓ clean
[DtiNet_Chemical_Chemical] ✓ clean
[STITCH_Chemical_Chemical] ✓ clean
[pheknowlator] ✓ clean
[hald] ✓ clean
[AgeAnnoMO] ✓ clean


## 3. Align Schemas and Concatenate

In [52]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'  # removed duplicate kg_source
]

df_list = []
for name, df in [
    ('Monarch_Chemical_Chemical',   Monarch_Chemical_Chemical),
    ('DRKG_Chemical_Chemical',      DRKG_Chemical_Chemical),
    ('PrimeKG_Chemical_Chemical_1', PrimeKG_Chemical_Chemical_1),
    ('PrimeKG_Chemical_Chemical_2', PrimeKG_Chemical_Chemical_2),
    ('PharmKG_Chemical_Chemical',   PharmKG_Chemical_Chemical),
    ('Hetionet_Chemical_Chemical',  Hetionet_Chemical_Chemical),
    ('CrossBAR_Chemical_Chemical',  CrossBAR_Chemical_Chemical),
    ('iBKH_Chemical_Chemical',      iBKH_Chemical_Chemical),
    ('DtiNet_Chemical_Chemical',    DtiNet_Chemical_Chemical),
    ('STITCH_Chemical_Chemical',    STITCH_Chemical_Chemical),
    ('pheknowlator',                pheknowlator),
    ('hald',                        hald),
    ('AgeAnnoMO',                   AgeAnnoMO)
]:
    dupes = df.columns[df.columns.duplicated()].tolist()
    if dupes:
        print(f"[{name}] duplicate columns found: {dupes}")
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df

Combined: 6,250,811 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,135398636,ChemicalEntity_ChemicalEntity,135823789,ChemicalEntity,related_to,ChemicalEntity,Monarch_KG,Pubchem,Pubchem,"guanosine 3',5'-bis(diphosphate)(5-)","guanosine 3',5'-bis(diphosphate)(6-)",Generalised,Homo sapiens
1,135398636,ChemicalEntity_ChemicalEntity,135398637,ChemicalEntity,related_to,ChemicalEntity,Monarch_KG,Pubchem,Pubchem,"guanosine 3',5'-bis(diphosphate)(5-)","guanosine 3',5'-bis(diphosphate)",Generalised,Homo sapiens
2,25201902,ChemicalEntity_ChemicalEntity,10100906,ChemicalEntity,related_to,ChemicalEntity,Monarch_KG,Pubchem,Pubchem,delphinidin 3-O-beta-D-glucoside-5-O-beta-D-gl...,delphinidin 3-O-beta-D-glucoside-5-O-beta-D-gl...,Generalised,Homo sapiens
3,86289381,ChemicalEntity_ChemicalEntity,54740357,ChemicalEntity,related_to,ChemicalEntity,Monarch_KG,Pubchem,Pubchem,stipitatate(1-),stipitatate(2-),Generalised,Homo sapiens
4,86289381,ChemicalEntity_ChemicalEntity,20501,ChemicalEntity,related_to,ChemicalEntity,Monarch_KG,Pubchem,Pubchem,stipitatate(1-),stipitatic acid,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6250806,154035,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,NaN,ChemicalEntity,ageannomo,Pubchem,Pubchem,"(2S,3S,4S,5R,6S)-3,4,5-trihydroxy-6-(4-methylp...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Aging,Homo sapiens
6250807,11902892,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,NaN,ChemicalEntity,ageannomo,Pubchem,Pubchem,"(2S,4R)-4-hydroxy-1-[(2S)-pyrrolidin-1-ium-2-c...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Aging,Homo sapiens
6250808,9085,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,NaN,ChemicalEntity,ageannomo,Pubchem,Pubchem,(2S)-2-amino-6-(diaminomethylideneamino)hexano...,"(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Aging,Homo sapiens
6250809,88299,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,NaN,ChemicalEntity,ageannomo,Pubchem,Pubchem,"N-(6-amino-3-methyl-2,4-dioxo-1H-pyrimidin-5-y...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Aging,Homo sapiens


## 4. Standardise Fixed-Value Columns

In [53]:
final_df['head_type'] = 'ChemicalEntity'
final_df['tail_type'] = 'ChemicalEntity'
final_df['relation']  = 'ChemicalEntity_ChemicalEntity'

# Ensure IDs are strings (removes float artefacts like '509.0')
final_df['head'] = final_df['head'].astype(str)
final_df['tail'] = final_df['tail'].astype(str)

print("Unique relation:",  set(final_df['relation']))
print("Unique head_type:", set(final_df['head_type']))
print("Unique tail_type:", set(final_df['tail_type']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'ChemicalEntity_ChemicalEntity'}
Unique head_type: {'ChemicalEntity'}
Unique tail_type: {'ChemicalEntity'}
Unique kg_source: {'Hetionet', 'ageannomo', 'ibkh', 'PrimeKG', 'dtinet', 'hald', 'stitch', 'pheknowlator', 'PharmKG', 'crossbar', 'DRKG', 'Monarch_KG'}


## 5. Deduplicate

Group by `(head, relation, tail)` and collapse `kg_source` with `::` separator.

In [55]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'head_id_is':       'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first',
    
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 6,250,811  |  After dedup: 3,370,334


,head,relation,tail,head_type,relation_type,tail_type,head_id_is,kg_source,kg_type,head_detail_name,tail_detail_name,species
0,1,ChemicalEntity_ChemicalEntity,104883,ChemicalEntity,None,ChemicalEntity,Pubchem,stitch,Generalised,3-acetyloxy-4-(trimethylazaniumyl)butanoate,tin(2+),Homo sapiens
1,1,ChemicalEntity_ChemicalEntity,1050,ChemicalEntity,None,ChemicalEntity,Pubchem,stitch,Generalised,3-acetyloxy-4-(trimethylazaniumyl)butanoate,3-hydroxy-5-(hydroxymethyl)-2-methylpyridine-4...,Homo sapiens
2,1,ChemicalEntity_ChemicalEntity,1051,ChemicalEntity,None,ChemicalEntity,Pubchem,stitch,Generalised,3-acetyloxy-4-(trimethylazaniumyl)butanoate,(4-formyl-5-hydroxy-6-methylpyridin-3-yl)methy...,Homo sapiens


## 6. Resolve Chemical Names

For all rows, derive `head_detail_name` / `tail_detail_name` and SMILES from:
1. **PubChem** IUPAC dict (numeric CIDs)
2. **DrugBank** name dict (IDs starting with `DB`)

Normalise DrugBank IDs to zero-padded 5-digit format (e.g. `DB5` → `DB00005`) before lookup.

In [56]:
# Step 1 – PubChem lookup (overwrites any pre-existing name columns)
final_df_dedup['head_detail_name'] = final_df_dedup['head'].map(Pubchem_IUPAC_CID_Dict)
final_df_dedup['head_smiles_name'] = final_df_dedup['head'].map(Pubchem_CID_Smile_Dict)
final_df_dedup['tail_detail_name'] = final_df_dedup['tail'].map(Pubchem_IUPAC_CID_Dict)
final_df_dedup['tail_smiles_name'] = final_df_dedup['tail'].map(Pubchem_CID_Smile_Dict)

# Step 2 – Normalise DrugBank IDs (DB\d+ → DB with 5-digit zero-padded number)
def standardize_drugbank_id(val):
    if isinstance(val, str):
        m = re.match(r'^DB(\d+)$', val)
        if m:
            return 'DB' + m.group(1).zfill(5)
    return val

final_df_dedup['head'] = final_df_dedup['head'].apply(standardize_drugbank_id)
final_df_dedup['tail'] = final_df_dedup['tail'].apply(standardize_drugbank_id)

# Step 3 – DrugBank fallback for DB-prefixed IDs still missing a name
for node in ['head', 'tail']:
    mask = final_df_dedup[f'{node}_detail_name'].isna() & final_df_dedup[node].str.startswith('DB')
    final_df_dedup.loc[mask, f'{node}_detail_name'] = final_df_dedup.loc[mask, node].map(Drugbank_dict)

# Step 4 – Drop rows where name could not be resolved for either node
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[
    ~final_df_dedup['head_detail_name'].isna() &
    ~final_df_dedup['tail_detail_name'].isna()
].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} unresolvable rows → {len(final_df_dedup):,} remaining")

Dropped 1,713 unresolvable rows → 3,368,621 remaining


## 7. Assign `head_id_is` / `tail_id_is`

- `DB`-prefixed IDs → `Drugbank`
- All others → `Pubchem`

In [58]:
for node in ['head', 'tail']:
    final_df_dedup[f'{node}_id_is'] = np.where(
        final_df_dedup[node].isna(), None,
        np.where(final_df_dedup[node].str.startswith('DB'), 'Drugbank', 'Pubchem')
    )

print("Unique head_id_is:", set(final_df_dedup['head_id_is']))
print("Unique tail_id_is:", set(final_df_dedup['tail_id_is']))

Unique head_id_is: {'Pubchem', 'Drugbank'}
Unique tail_id_is: {'Pubchem', 'Drugbank'}


In [63]:
print("kg_source values present:", (final_df_dedup['kg_source'].value_counts()), final_df_dedup['kg_type'].value_counts())

kg_source values present: kg_source
ibkh                                                   832069
DRKG::PrimeKG::ibkh                                    770569
PrimeKG::ibkh                                          546657
stitch                                                 502062
DRKG::ibkh                                             494692
                                                        ...  
PharmKG::hald                                               1
Monarch_KG::PrimeKG::crossbar::ibkh                         1
PrimeKG::crossbar::ibkh::pheknowlator                       1
DRKG::PrimeKG::crossbar::ibkh::pheknowlator::stitch         1
DRKG::PrimeKG::dtinet::hald::ibkh                           1
Name: count, Length: 112, dtype: int64 kg_type
Generalised           3367790
Aging                     782
Aging::Generalised         49
Name: count, dtype: int64


In [62]:
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,head_id_is,kg_source,kg_type,head_detail_name,tail_detail_name,species,head_smiles_name,tail_smiles_name,tail_id_is
0,1,ChemicalEntity_ChemicalEntity,104883,ChemicalEntity,None,ChemicalEntity,Pubchem,stitch,Generalised,3-acetyloxy-4-(trimethylazaniumyl)butanoate,tin(2+),Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,[Sn+2],Pubchem
1,1,ChemicalEntity_ChemicalEntity,1050,ChemicalEntity,None,ChemicalEntity,Pubchem,stitch,Generalised,3-acetyloxy-4-(trimethylazaniumyl)butanoate,3-hydroxy-5-(hydroxymethyl)-2-methylpyridine-4...,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,CC1=NC=C(C(=C1O)C=O)CO,Pubchem
2,1,ChemicalEntity_ChemicalEntity,1051,ChemicalEntity,None,ChemicalEntity,Pubchem,stitch,Generalised,3-acetyloxy-4-(trimethylazaniumyl)butanoate,(4-formyl-5-hydroxy-6-methylpyridin-3-yl)methy...,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,CC1=NC=C(C(=C1O)C=O)COP(=O)(O)O,Pubchem
3,1,ChemicalEntity_ChemicalEntity,1052,ChemicalEntity,None,ChemicalEntity,Pubchem,stitch,Generalised,3-acetyloxy-4-(trimethylazaniumyl)butanoate,4-(aminomethyl)-5-(hydroxymethyl)-2-methylpyri...,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,CC1=NC=C(C(=C1O)CN)CO,Pubchem
4,1,ChemicalEntity_ChemicalEntity,1054,ChemicalEntity,None,ChemicalEntity,Pubchem,stitch,Generalised,3-acetyloxy-4-(trimethylazaniumyl)butanoate,"4,5-bis(hydroxymethyl)-2-methylpyridin-3-ol",Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,CC1=NC=C(C(=C1O)CO)CO,Pubchem
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3368616,DB16019,ChemicalEntity_ChemicalEntity,3397,ChemicalEntity,None,ChemicalEntity,Drugbank,ibkh,Generalised,Gallium Ga-68 gozetotide,2-methyl-N-[4-nitro-3-(trifluoromethyl)phenyl]...,Homo sapiens,NaN,CC(C)C(=O)NC1=CC(=C(C=C1)[N+](=O)[O-])C(F)(F)F,Pubchem
3368617,DB16019,ChemicalEntity_ChemicalEntity,4493,ChemicalEntity,None,ChemicalEntity,Drugbank,ibkh,Generalised,Gallium Ga-68 gozetotide,"5,5-dimethyl-3-[4-nitro-3-(trifluoromethyl)phe...",Homo sapiens,NaN,CC1(C(=O)N(C(=O)N1)C2=CC(=C(C=C2)[N+](=O)[O-])...,Pubchem
3368618,DB16019,ChemicalEntity_ChemicalEntity,5284533,ChemicalEntity,None,ChemicalEntity,Drugbank,ibkh,Generalised,Gallium Ga-68 gozetotide,"(8R,9S,10R,13S,14S,17R)-17-acetyl-6-chloro-17-...",Homo sapiens,NaN,CC(=O)[C@]1(CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2C=C(...,Pubchem
3368619,DB16019,ChemicalEntity_ChemicalEntity,67171867,ChemicalEntity,None,ChemicalEntity,Drugbank,ibkh,Generalised,Gallium Ga-68 gozetotide,N-[(2S)-1-[3-(3-chloro-4-cyanophenyl)pyrazol-1...,Homo sapiens,NaN,C[C@@H](CN1C=CC(=N1)C2=CC(=C(C=C2)C#N)Cl)NC(=O...,Pubchem


## 8. Add Schema Columns and Enforce Column Order

In [64]:

# SMILES is an extra column beyond REQUIRED_COLS — keep it appended at the end
EXTRA_COLS = ['head_smiles_name', 'tail_smiles_name']
final_df_dedup = final_df_dedup[REQUIRED_COLS + EXTRA_COLS]

print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (3368621, 15)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name,tail_smiles_name
0,1,ChemicalEntity_ChemicalEntity,104883,ChemicalEntity,None,ChemicalEntity,stitch,Pubchem,Pubchem,3-acetyloxy-4-(trimethylazaniumyl)butanoate,tin(2+),Generalised,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,[Sn+2]
1,1,ChemicalEntity_ChemicalEntity,1050,ChemicalEntity,None,ChemicalEntity,stitch,Pubchem,Pubchem,3-acetyloxy-4-(trimethylazaniumyl)butanoate,3-hydroxy-5-(hydroxymethyl)-2-methylpyridine-4...,Generalised,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,CC1=NC=C(C(=C1O)C=O)CO
2,1,ChemicalEntity_ChemicalEntity,1051,ChemicalEntity,None,ChemicalEntity,stitch,Pubchem,Pubchem,3-acetyloxy-4-(trimethylazaniumyl)butanoate,(4-formyl-5-hydroxy-6-methylpyridin-3-yl)methy...,Generalised,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,CC1=NC=C(C(=C1O)C=O)COP(=O)(O)O
3,1,ChemicalEntity_ChemicalEntity,1052,ChemicalEntity,None,ChemicalEntity,stitch,Pubchem,Pubchem,3-acetyloxy-4-(trimethylazaniumyl)butanoate,4-(aminomethyl)-5-(hydroxymethyl)-2-methylpyri...,Generalised,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,CC1=NC=C(C(=C1O)CN)CO
4,1,ChemicalEntity_ChemicalEntity,1054,ChemicalEntity,None,ChemicalEntity,stitch,Pubchem,Pubchem,3-acetyloxy-4-(trimethylazaniumyl)butanoate,"4,5-bis(hydroxymethyl)-2-methylpyridin-3-ol",Generalised,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,CC1=NC=C(C(=C1O)CO)CO


## 9. QC — NaN Check

In [65]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })
nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,1390789,0,1390789
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [66]:
print("\nkg_source values present:", set(final_df_dedup['kg_source']))


kg_source values present: {'DRKG::dtinet::ibkh::stitch', 'PharmKG::crossbar::pheknowlator', 'crossbar::ibkh', 'DRKG::Hetionet::PrimeKG::crossbar::ibkh', 'hald', 'PharmKG::crossbar', 'PrimeKG::hald::ibkh', 'DRKG::PrimeKG::ibkh', 'DRKG::Hetionet::dtinet::ibkh::stitch', 'PrimeKG::crossbar::ibkh::pheknowlator::stitch', 'DRKG::PrimeKG::dtinet::hald::ibkh', 'Monarch_KG::crossbar::pheknowlator', 'dtinet::ibkh', 'PrimeKG::ibkh::stitch', 'hald::ibkh::stitch', 'DRKG::Hetionet::hald::ibkh', 'DRKG::PrimeKG::ibkh::pheknowlator::stitch', 'DRKG::Hetionet::crossbar::ibkh', 'DRKG::PharmKG::PrimeKG::dtinet::ibkh', 'DRKG::ibkh::stitch', 'DRKG::Hetionet::ibkh', 'DRKG::Monarch_KG::PrimeKG::crossbar::ibkh::pheknowlator', 'DRKG::Hetionet::PrimeKG::ibkh::stitch', 'PharmKG::ibkh::stitch', 'ageannomo', 'PrimeKG::ibkh', 'dtinet::ibkh::stitch', 'crossbar::ibkh::pheknowlator', 'PharmKG::stitch', 'DRKG::PharmKG::PrimeKG::ibkh::stitch', 'Monarch_KG::crossbar::stitch', 'DRKG::Hetionet::PrimeKG::hald::ibkh', 'Monarch

## 10. Save Output

In [67]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 3,368,621 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_CHEMICALENTITY/ALL_CHEMICALENTITY_CHEMICALENTITY.csv
